## Bayesian XG Model 
### by Adam Sequeira 
- put arviz paper here

#### Problem Statement 
In football there is a varying scale of richness of tagging data, if paying for third party systems, you can get or generate at a minimum basic (x,y) co-ordinates. From this you have the ability to generate a baseline xG model, by calculating shot_distance and shot_angle. 

One step further would be data that includes features associated with shots, including what body_part was used, was the player under_pressure, the shot technique, the play_pattern (free_kick, penalty etc...), this was standard in statsbomb's open dataset 

One step further than that, has been the release of statsbomb including freeze-frame data as part of their events, this include the x,y position of players off the ball along with other features like the assist_type and assist_height. 

While richer datasets generally produce more accurate xG estimates, the inverse is also true: when fewer features are available, there is greater uncertainty regarding the true probability of a goal. Traditional xG models, commonly implemented using logistic regression or machine learning classifiers, typically return a single point estimate for each shot. Although useful, these point estimates do not communicate how certain the model is about its prediction.

This creates a challenge when comparing xG values generated from different data providers, competitions, or historical datasets that contain varying levels of information. Two shots may receive the same predicted xG value despite one being estimated from rich contextual data and the other from only basic location data. Intuitively, confidence in these estimates should differ.

Bayesian modeling provides a natural framework for addressing this problem. Rather than producing a single probability estimate, Bayesian models generate a posterior distribution over the goal probability. This distribution quantifies both the estimated probability of scoring and the uncertainty surrounding that estimate. When informative features are unavailable, posterior distributions can widen to reflect increased uncertainty. Conversely, when richer contextual information is available, the posterior distribution can become more concentrated, indicating greater confidence in the prediction.

#### Why is this Relevant? 
The ability to quantify uncertainty is valuable for both practitioners and researchers. In football analytics, model outputs are often used to evaluate player/match shot quality. In recruitment, players go through different systems with different levels of richness in their data. 

A probabilistic framework also allows analysts to:

Distinguish between aleatoric uncertainty (inherent randomness in goal scoring) and epistemic uncertainty (uncertainty arising from limited information).
Compare xG estimates across datasets with different levels of feature richness.
Incorporate uncertainty directly into downstream models and decision-making processes.
Produce more robust estimates in situations with sparse data or rare events.


#### Goal 
- Generate one size fits all bayesian xg model that provides the following
    - posterior distribution of outcomes instead of standard point estimates 
    - able to factor in error for different tagging systems. 

In [1]:
import nutpie
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from pathlib import Path

### Data Analysis 

#### Model Builder 
Below is a form of model builder used in pymc, including relevant transformations,
continuous values are scaled using standard scaler.  

In [2]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

class BayesianXg:
    """
    Bayesian expected goals model.

    Sampling options
    ----------------
    method="nuts"   : standard PyMC NUTS (default)
    method="advi"   : variational inference (fast)
    method="nutpie" : nutpie NUTS sampler (fast, requires nutpie)
    """

    version = "0.1"

    MODEL_VARIABLES = {
        "cont_cols": [
            "shot_angle",
            "shot_distance",
            "shot_distance_angle",
            "goalkeeper_shooter_distance",
            "goalkeeper_shooter_angle",
            "goalkeeper_shooter_distance_angle",
            "players_between_shooter_and_goal",
        ],
        "bin_cols": [
            "shot_first_time",
            "shot_follows_dribble",
            "under_pressure",
            "is_open_goal",
        ],
        "cat_cols": [
            "shot_body_part",
            "shot_technique",
            "assist_type",
            "assist_height",
            "play_pattern",
        ],
        "pressure_col": "defensive_pressure",
    }

    def __init__(
            self, 
            df: pd.DataFrame = None, 
            target_col: str = None, 
            method: str = "nutpie", 
            **sample_kwargs
        ):

        self.idata = None
        self.approx = None          # set when method="advi"
        self._scaler = None         # fit on training data, reused at inference
        self._cat_categories = None
        self._pressure_levels = None

        self.method = method
        self.target_col = target_col if target_col else None
        self._model_inputs = self._preprocess(df, target_col, fit_scaler=True) if df is not None and target_col is not None else None
        self.model = self._build_model(model_inputs = self._model_inputs) if self._model_inputs is not None else None
        
    # =========================================================
    # PUBLIC API
    # =========================================================

    def fit(self, df: pd.DataFrame = None, target_col: str = None, method: str = None, **sample_kwargs):
        """
        Build and sample the model.

        Parameters
        ----------
        df          : training dataframe (must contain target_col)
        target_col  : name of the 0/1 goal column
        method      : "nuts" | "advi" | "nutpie"
        **sample_kwargs : passed through to the sampler
        """
        if not method:
            method = self.method

        if not self._model_inputs:
            model_inputs = self._preprocess(df, target_col, fit_scaler=True)
            self._model_inputs = model_inputs
            self.model = self._build_model(model_inputs = self._model_inputs)

        if method == "advi":
            self.idata, self.approx = self._sample_advi(self.model, **sample_kwargs)
        elif method == "nutpie":
            self.idata = self._sample_nutpie(self.model, **sample_kwargs)
        elif method == "nuts":
            self.idata = self._sample_nuts(self.model, **sample_kwargs)
        else:
            raise ValueError(f"Unknown method '{method}'. Choose 'nuts', 'advi', or 'nutpie'.")

        return self

    def predict(self, df: pd.DataFrame, target_col: str = None):
        """
        Return posterior mean xG and std for each row in df.
        Requires fit() to have been called first.
        """
        if self.idata is None:
            raise RuntimeError("Call fit() before predict().")

        if not target_col:
            target_col = self.target_col
        
        feats = self._preprocess(df, target_col=None, fit_scaler=False)
        xg_mean, xg_std = self._posterior_mean_xg(df, target_col, self.idata.posterior)
        return xg_mean, xg_std

    def evaluate(
            self,
            df: pd.DataFrame,
            target_col: str,
            use_posterior_predict_sampling: bool = False,
            n_samples: int = 1000,
        ):
        """Return ROC-AUC and Brier score on df."""
        if use_posterior_predict_sampling:
            ppc = self._posterior_predict_sampling(df, target_col=target_col, n_samples=n_samples)

            if hasattr(ppc, "posterior_predictive"):
                p_samples = np.asarray(ppc.posterior_predictive["p"].values)
            elif isinstance(ppc, dict):
                p_samples = np.asarray(ppc["p"])
            else:
                raise TypeError("Unsupported posterior predictive return type.")

            # Flatten sample dimensions while preserving shot axis.
            p_samples = p_samples.reshape(-1, p_samples.shape[-1])
            if n_samples is not None and p_samples.shape[0] > n_samples:
                p_samples = p_samples[:n_samples]

            xg_mean = p_samples.mean(axis=0)
            xg_std = p_samples.std(axis=0)
        else:
            xg_mean, xg_std = self.predict(df, target_col=target_col)

        y_true = df[target_col].fillna(0).astype(int).to_numpy()
        return {
            "roc_auc": roc_auc_score(y_true, xg_mean),
            "brier":   brier_score_loss(y_true, xg_mean),
            "xg_mean": xg_mean,
            "xg_std":  xg_std,
        }

    def save_trace(self, path: str):
        """Save InferenceData to netCDF4 (.nc)."""
        if self.idata is None:
            raise RuntimeError("Nothing to save — call fit() first.")
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        self.idata.to_netcdf(str(p))
        print(f"Trace saved → {p}")

    def load_trace(self, path: str):
        """Load a previously saved InferenceData."""
        self.idata = az.from_netcdf(path)
        print(f"Trace loaded ← {path}")
        return self

    # =========================================================
    # SAMPLERS
    # =========================================================

    def _sample_nuts(self, model, draws=1000, tune=1000, chains=4, target_accept=0.9, **kwargs):
        with model:
            idata = pm.sample(
                draws=draws,
                tune=tune,
                chains=chains,
                target_accept=target_accept,
                progressbar=True,
                **kwargs,
            )
        return idata

    def _sample_advi(self, model, n=50_000, draws=2000, **kwargs):
        with model:
            approx = pm.fit(
                n=n,
                method="advi",
                progressbar=True,
                **kwargs,
            )
            idata = approx.sample(draws)
        return idata, approx

    def _sample_nutpie(self, model, draws=1000, tune=1000, chains=4, **kwargs):
        compiled = nutpie.compile_pymc_model(model)
        idata = nutpie.sample(
            compiled,
            draws=draws,
            tune=tune,
            chains=chains,
            **kwargs,
        )
        return idata

    # =========================================================
    # MODEL DEFINITION
    # =========================================================

    def _build_model(self, model_inputs: dict = None):
        
        if not model_inputs:
            model_inputs = self._model_inputs

        with pm.Model() as model:

            # --- Continuous Values ---
            shot_angle     = pm.Data("shot_angle",     np.asarray(model_inputs["shot_angle"]))
            shot_distance  = pm.Data("shot_distance",  np.asarray(model_inputs["shot_distance"]))
            shot_dist_ang  = pm.Data("shot_distance_angle", np.asarray(model_inputs["shot_distance_angle"]))
            gk_dist        = pm.Data("goalkeeper_shooter_distance",       np.asarray(model_inputs["goalkeeper_shooter_distance"]))
            gk_angle       = pm.Data("goalkeeper_shooter_angle",          np.asarray(model_inputs["goalkeeper_shooter_angle"]))
            gk_dist_ang    = pm.Data("goalkeeper_shooter_distance_angle", np.asarray(model_inputs["goalkeeper_shooter_distance_angle"]))
            players_bet    = pm.Data("players_between", np.asarray(model_inputs["players_between_shooter_and_goal"]))

            # --- Binary Values ---
            shot_ft  = pm.Data("shot_first_time",      np.asarray(model_inputs["shot_first_time"]))
            shot_fd  = pm.Data("shot_follows_dribble", np.asarray(model_inputs["shot_follows_dribble"]))
            under_p  = pm.Data("under_pressure",       np.asarray(model_inputs["under_pressure"]))
            open_g   = pm.Data("is_open_goal",         np.asarray(model_inputs["is_open_goal"]))

            # --- Continuous Missing Masks ---
            shot_angle_m    = pm.Data("shot_angle_missing",          np.asarray(model_inputs["shot_angle_missing"]))
            shot_dist_m     = pm.Data("shot_distance_missing",       np.asarray(model_inputs["shot_distance_missing"]))
            shot_dist_ang_m = pm.Data("shot_distance_angle_missing", np.asarray(model_inputs["shot_distance_angle_missing"]))
            gk_dist_m       = pm.Data("gk_dist_missing",             np.asarray(model_inputs["goalkeeper_shooter_distance_missing"]))
            gk_angle_m      = pm.Data("gk_angle_missing",            np.asarray(model_inputs["goalkeeper_shooter_angle_missing"]))
            gk_dist_ang_m   = pm.Data("gk_dist_angle_missing",       np.asarray(model_inputs["goalkeeper_shooter_distance_angle_missing"]))
            players_bet_m   = pm.Data("players_between_missing",     np.asarray(model_inputs["players_between_shooter_and_goal_missing"]))

            # --- Binary Missing Masks ---
            ft_m  = pm.Data("shot_first_time_missing",      np.asarray(model_inputs["shot_first_time_missing"]))
            fd_m  = pm.Data("shot_follows_dribble_missing", np.asarray(model_inputs["shot_follows_dribble_missing"]))
            up_m  = pm.Data("under_pressure_missing",       np.asarray(model_inputs["under_pressure_missing"]))
            oog_m = pm.Data("is_open_goal_missing",         np.asarray(model_inputs["is_open_goal_missing"]))

            # --- Cat indices ---
            sbp_idx = pm.Data("shot_body_part_idx", np.asarray(model_inputs["cat_idx"]["shot_body_part"]))
            st_idx  = pm.Data("shot_technique_idx", np.asarray(model_inputs["cat_idx"]["shot_technique"]))
            at_idx  = pm.Data("assist_type_idx",    np.asarray(model_inputs["cat_idx"]["assist_type"]))
            ah_idx  = pm.Data("assist_height_idx",  np.asarray(model_inputs["cat_idx"]["assist_height"]))
            pp_idx  = pm.Data("play_pattern_idx",   np.asarray(model_inputs["cat_idx"]["play_pattern"]))

            # --- Pressure ---
            dp_idx = pm.Data("pressure_idx",     np.asarray(model_inputs["pressure_idx"]))
            dp_m   = pm.Data("pressure_missing", np.asarray(model_inputs["pressure_missing"]))

            y = pm.Data("y", np.asarray(model_inputs["y"]))

            # =====================================================
            # PRIORS
            # =====================================================

            # intercept anchored to ~10% base rate (logit(0.10) ≈ -2.2)
            # alpha = pm.Normal("alpha", mu=-2.2, sigma=1)
            alpha = 0

            # embeddings
            sbp_emb = pm.Normal("shot_body_part_emb", 0, 5, shape=model_inputs["n_levels"]["shot_body_part"])
            st_emb  = pm.Normal("shot_technique_emb", 0, 5, shape=model_inputs["n_levels"]["shot_technique"])
            at_emb  = pm.Normal("assist_type_emb",    0, 5, shape=model_inputs["n_levels"]["assist_type"])
            ah_emb  = pm.Normal("assist_height_emb",  0, 5, shape=model_inputs["n_levels"]["assist_height"])
            pp_emb  = pm.Normal("play_pattern_emb",   0, 5, shape=model_inputs["n_levels"]["play_pattern"])

            # continuous betas
            b_shot_angle      = pm.SkewNormal("beta_shot_angle",          mu=0,    sigma=5, alpha=1)
            b_shot_dist       = pm.SkewNormal("beta_shot_distance",       mu=0,    sigma=5, alpha=-1)
            b_shot_dist_angle = pm.SkewNormal("beta_shot_distance_angle", mu=0.5,  sigma=1, alpha=4)
            b_gk_dist         = pm.SkewNormal("beta_gk_dist",             mu=0,    sigma=5, alpha=1)
            b_gk_angle        = pm.SkewNormal("beta_gk_angle",            mu=1,    sigma=1, alpha=4)
            b_gk_dist_angle   = pm.SkewNormal("beta_gk_dist_angle",       mu=-1,   sigma=1, alpha=-4)
            b_players         = pm.SkewNormal("beta_players_between",     mu=-1,   sigma=1.5, alpha=-6)

            # binary betas
            b_ft  = pm.Normal("beta_shot_first_time",      mu=0, sigma=5)
            b_fd  = pm.Normal("beta_shot_follows_dribble", mu=0, sigma=5)
            b_up  = pm.Normal("beta_under_pressure",       mu=0, sigma=5)
            b_og  = pm.SkewNormal("beta_is_open_goal",     mu=0, sigma=5, alpha=4)

            # shared missing signals
            b_missing     = pm.Normal("beta_missing",     mu=0, sigma=1)
            b_missing_cat = pm.Normal("beta_missing_cat", mu=0, sigma=1)

            # pressure
            b_pressure     = pm.Normal("beta_pressure_levels", mu=0, sigma=1, shape=model_inputs["n_pressure_levels"])
            b_pressure_m   = pm.Normal("beta_pressure_missing", mu=0, sigma=1)

            # =====================================================
            # LINEAR PREDICTOR
            # =====================================================

            mu_lin = alpha

            # continuous (zeroed when missing)
            mu_lin += b_shot_angle      * shot_angle   * (1 - shot_angle_m)
            mu_lin += b_shot_dist       * shot_distance * (1 - shot_dist_m)
            mu_lin += b_shot_dist_angle * shot_dist_ang * (1 - shot_dist_ang_m)
            mu_lin += b_gk_dist         * gk_dist       * (1 - gk_dist_m)
            mu_lin += b_gk_angle        * gk_angle      * (1 - gk_angle_m)
            mu_lin += b_gk_dist_angle   * gk_dist_ang   * (1 - gk_dist_ang_m)
            mu_lin += b_players         * players_bet   * (1 - players_bet_m)

            # binary (zeroed when missing)
            mu_lin += b_ft * shot_ft * (1 - ft_m)
            mu_lin += b_fd * shot_fd * (1 - fd_m)
            mu_lin += b_up * under_p * (1 - up_m)
            mu_lin += b_og * open_g  * (1 - oog_m)

            # shared missing signal: continuous + binary
            mu_lin += b_missing * (
                shot_angle_m + shot_dist_m + shot_dist_ang_m +
                gk_dist_m + gk_angle_m + gk_dist_ang_m +
                players_bet_m +
                ft_m + fd_m + up_m + oog_m
            )

            # pressure
            mu_lin += b_pressure[dp_idx] * (1 - dp_m) + b_pressure_m * dp_m

            # categorical embeddings (present) + shared missing signal
            sbp_miss = np.asarray(model_inputs["shot_body_part_missing"])
            st_miss  = np.asarray(model_inputs["shot_technique_missing"])
            at_miss  = np.asarray(model_inputs["assist_type_missing"])
            ah_miss  = np.asarray(model_inputs["assist_height_missing"])
            pp_miss  = np.asarray(model_inputs["play_pattern_missing"])

            mu_lin += sbp_emb[sbp_idx] * (1 - sbp_miss)
            mu_lin += st_emb[st_idx]   * (1 - st_miss)
            mu_lin += at_emb[at_idx]   * (1 - at_miss)
            mu_lin += ah_emb[ah_idx]   * (1 - ah_miss)
            mu_lin += pp_emb[pp_idx]   * (1 - pp_miss)

            mu_lin += b_missing_cat * (sbp_miss + st_miss + at_miss + ah_miss + pp_miss)

            # =====================================================
            # LIKELIHOOD
            # =====================================================

            p = pm.Deterministic("p", pm.math.sigmoid(mu_lin))
            pm.Bernoulli("y_obs", p=p, observed=y)

        return model
    
    def _posterior_predict_sampling(self, df: pd.DataFrame, target_col: str, n_samples: int = 1000):
        """
        Compute posterior predictive samples by sampling from the posterior distribution of parameters.
        This would involve drawing n_samples sets of parameters from the posterior, computing the 
        linear predictor for each set, and then applying the sigmoid function to get predicted probabilities.
        The result would be an (n_samples, N) array of predicted probabilities for each shot.
        """

        model_inputs = self._preprocess(df, target_col=target_col, fit_scaler=True)
        xg_model = self._build_model(model_inputs = model_inputs)
        idata = self.idata

        with xg_model:
            pm.set_data({
                # =====================================================
                # CONTINUOUS
                # =====================================================
                "shot_angle": np.asarray(model_inputs["shot_angle"]),
                "shot_distance": np.asarray(model_inputs["shot_distance"]),
                "shot_distance_angle": np.asarray(model_inputs["shot_distance_angle"]),

                "goalkeeper_shooter_distance": np.asarray(model_inputs["goalkeeper_shooter_distance"]),
                "goalkeeper_shooter_angle": np.asarray(model_inputs["goalkeeper_shooter_angle"]),
                "goalkeeper_shooter_distance_angle": np.asarray(model_inputs["goalkeeper_shooter_distance_angle"]),

                "players_between": np.asarray(
                    model_inputs["players_between_shooter_and_goal"]
                ),

                # =====================================================
                # CONTINUOUS MISSING
                # =====================================================
                "shot_angle_missing": np.asarray(model_inputs["shot_angle_missing"]),
                "shot_distance_missing": np.asarray(model_inputs["shot_distance_missing"]),
                "shot_distance_angle_missing": np.asarray(model_inputs["shot_distance_angle_missing"]),

                "gk_dist_missing": np.asarray(
                    model_inputs["goalkeeper_shooter_distance_missing"]
                ),
                "gk_angle_missing": np.asarray(
                    model_inputs["goalkeeper_shooter_angle_missing"]
                ),
                "gk_dist_angle_missing": np.asarray(
                    model_inputs["goalkeeper_shooter_distance_angle_missing"]
                ),

                "players_between_missing": np.asarray(
                    model_inputs["players_between_shooter_and_goal_missing"]
                ),

                # =====================================================
                # BINARY
                # =====================================================
                "shot_first_time": np.asarray(model_inputs["shot_first_time"]),
                "shot_follows_dribble": np.asarray(model_inputs["shot_follows_dribble"]),
                "under_pressure": np.asarray(model_inputs["under_pressure"]),
                "is_open_goal": np.asarray(model_inputs["is_open_goal"]),

                # =====================================================
                # BINARY MISSING
                # =====================================================
                "shot_first_time_missing": np.asarray(model_inputs["shot_first_time_missing"]),
                "shot_follows_dribble_missing": np.asarray(model_inputs["shot_follows_dribble_missing"]),
                "under_pressure_missing": np.asarray(model_inputs["under_pressure_missing"]),
                "is_open_goal_missing": np.asarray(model_inputs["is_open_goal_missing"]),

                # =====================================================
                # PRESSURE
                # =====================================================
                "pressure_idx": np.asarray(model_inputs["pressure_idx"]),
                "pressure_missing": np.asarray(model_inputs["pressure_missing"]),

                # =====================================================
                # CATEGORICAL
                # =====================================================
                "shot_body_part_idx": np.asarray(model_inputs["cat_idx"]["shot_body_part"]),
                "shot_technique_idx": np.asarray(model_inputs["cat_idx"]["shot_technique"]),
                "assist_type_idx": np.asarray(model_inputs["cat_idx"]["assist_type"]),
                "assist_height_idx": np.asarray(model_inputs["cat_idx"]["assist_height"]),
                "play_pattern_idx": np.asarray(model_inputs["cat_idx"]["play_pattern"]),

                # # =====================================================
                # # CATEGORICAL MISSING
                # # =====================================================
                # "shot_body_part_missing": np.asarray(model_inputs["shot_body_part_missing"]),
                # "shot_technique_missing": np.asarray(model_inputs["shot_technique_missing"]),
                # "assist_type_missing": np.asarray(model_inputs["assist_type_missing"]),
                # "assist_height_missing": np.asarray(model_inputs["assist_height_missing"]),
                # "play_pattern_missing": np.asarray(model_inputs["play_pattern_missing"]),

                # =====================================================
                # TARGET (dummy)
                # =====================================================
                "y": np.asarray(model_inputs["y"]),
            })

            ppc = pm.sample_posterior_predictive(
                idata,
                var_names=["y_obs", "p"],
                random_seed=42
            )

        return ppc

    # =========================================================
    # POSTERIOR INFERENCE (numpy, no PyMC context needed)
    # =========================================================

    def _posterior_mean_xg(self, df: pd.DataFrame, target_col: str, posterior):
        """
        Compute posterior mean xG from features and an ArviZ posterior group.
        Returns (xg_mean, xg_std), each shape (N,).
        """

        def scalar(name):
            return np.asarray(posterior[name].values).reshape(-1)       # (S,)

        def emb(name):
            arr = np.asarray(posterior[name].values)
            return arr.reshape(-1, *arr.shape[2:])                 # (S, levels)
        
        inputs = self._preprocess(df, target_col=target_col, fit_scaler=False)
        S = scalar("beta_shot_angle").shape[0]
        N = inputs["shot_angle"].shape[0]

        # helpers: explicit shapes eliminate any broadcasting ambiguity
        def b(arr): return arr.reshape(S, 1)    # (S, 1)
        def f(vec): return vec.reshape(1, N)    # (1, N)

        mu = np.zeros((S, N), dtype=float)

        # continuous
        mu += b(scalar("beta_shot_angle"))          * f(inputs["shot_angle"] * (1 - inputs["shot_angle_missing"]))
        mu += b(scalar("beta_shot_distance"))       * f(inputs["shot_distance"] * (1 - inputs["shot_distance_missing"]))
        mu += b(scalar("beta_shot_distance_angle")) * f(inputs["shot_distance_angle"] * (1 - inputs["shot_distance_angle_missing"]))
        mu += b(scalar("beta_gk_dist"))             * f(inputs["goalkeeper_shooter_distance"] * (1 - inputs["goalkeeper_shooter_distance_missing"]))
        mu += b(scalar("beta_gk_angle"))            * f(inputs["goalkeeper_shooter_angle"] * (1 - inputs["goalkeeper_shooter_angle_missing"]))
        mu += b(scalar("beta_gk_dist_angle"))       * f(inputs["goalkeeper_shooter_distance_angle"] * (1 - inputs["goalkeeper_shooter_distance_angle_missing"]))
        mu += b(scalar("beta_players_between"))     * f(inputs["players_between_shooter_and_goal"] * (1 - inputs["players_between_shooter_and_goal_missing"]))

        # binary
        mu += b(scalar("beta_shot_first_time"))      * f(inputs["shot_first_time"] * (1 - inputs["shot_first_time_missing"]))
        mu += b(scalar("beta_shot_follows_dribble")) * f(inputs["shot_follows_dribble"] * (1 - inputs["shot_follows_dribble_missing"]))
        mu += b(scalar("beta_under_pressure"))       * f(inputs["under_pressure"] * (1 - inputs["under_pressure_missing"]))
        mu += b(scalar("beta_is_open_goal"))         * f(inputs["is_open_goal"] * (1 - inputs["is_open_goal_missing"]))

        # shared missing signal: continuous + binary
        all_missing = (
            inputs["shot_angle_missing"]
            + inputs["shot_distance_missing"]
            + inputs["shot_distance_angle_missing"]
            + inputs["goalkeeper_shooter_distance_missing"]
            + inputs["goalkeeper_shooter_angle_missing"]
            + inputs["goalkeeper_shooter_distance_angle_missing"]
            + inputs["players_between_shooter_and_goal_missing"]
            + inputs["shot_first_time_missing"]
            + inputs["shot_follows_dribble_missing"]
            + inputs["under_pressure_missing"]
            + inputs["is_open_goal_missing"]
        )
        mu += b(scalar("beta_missing")) * f(all_missing)

        # pressure
        b_pres = emb("beta_pressure_levels")                   # (S, n_levels)
        mu += b_pres[:, inputs["pressure_idx"]] * f(1 - inputs["pressure_missing"])
        mu += b(scalar("beta_pressure_missing")) * f(inputs["pressure_missing"])

        # categorical
        b_missing_cat = scalar("beta_missing_cat")
        cat_miss_total = np.zeros(N)
        for col, e_name in [
            ("shot_body_part", "shot_body_part_emb"),
            ("shot_technique",  "shot_technique_emb"),
            ("assist_type",     "assist_type_emb"),
            ("assist_height",   "assist_height_emb"),
            ("play_pattern",    "play_pattern_emb"),
        ]:
            e   = emb(e_name)                   # (S, n_levels)
            idx = inputs["cat_idx"][col]                  # (N,)
            miss = inputs["cat_missing"][col]             # (N,)
            mu += e[:, idx] * f(1 - miss)
            cat_miss_total += miss

        mu += b(b_missing_cat) * f(cat_miss_total)

        p = sigmoid(mu)
        return p.mean(axis=0), p.std(axis=0)

    # =========================================================
    # PREPROCESSING
    # =========================================================

    def _preprocess(self, df: pd.DataFrame, target_col, fit_scaler: bool) -> dict:
        mv = self.MODEL_VARIABLES
        X = df.copy()

        # derived features
        if "shot_distance_angle" not in X.columns:
            X["shot_distance_angle"] = X["shot_angle"] * X["shot_distance"]
        if "goalkeeper_shooter_distance_angle" not in X.columns:
            X["goalkeeper_shooter_distance_angle"] = (
                X["goalkeeper_shooter_angle"] * X["goalkeeper_shooter_distance"]
            )

        # ensure all expected columns exist
        for col in mv["cont_cols"] + mv["bin_cols"] + mv["cat_cols"] + [mv["pressure_col"]]:
            if col not in X.columns:
                X[col] = np.nan

        X_cont, X_cont_missing = self._transform_continuous(X, fit_scaler)
        X_bin,  X_bin_missing  = self._transform_binary(X)
        cat_idx, cat_missing, cat_categories, n_levels = self._transform_categorical(X, fit_scaler)
        pressure_idx, pressure_missing, n_pressure_levels, levels = self._transform_pressure(X, fit_scaler)

        model_inputs = {
            # continuous
            "shot_angle":                          X_cont[:, 0],
            "shot_angle_missing":                  X_cont_missing[:, 0],
            "shot_distance":                       X_cont[:, 1],
            "shot_distance_missing":               X_cont_missing[:, 1],
            "shot_distance_angle":                 X_cont[:, 2],
            "shot_distance_angle_missing":         X_cont_missing[:, 2],
            "goalkeeper_shooter_distance":         X_cont[:, 3],
            "goalkeeper_shooter_distance_missing": X_cont_missing[:, 3],
            "goalkeeper_shooter_angle":            X_cont[:, 4],
            "goalkeeper_shooter_angle_missing":    X_cont_missing[:, 4],
            "goalkeeper_shooter_distance_angle":   X_cont[:, 5],
            "goalkeeper_shooter_distance_angle_missing": X_cont_missing[:, 5],
            "players_between_shooter_and_goal":        X_cont[:, 6],
            "players_between_shooter_and_goal_missing": X_cont_missing[:, 6],
            # binary
            "shot_first_time":           X_bin[:, 0],
            "shot_first_time_missing":   X_bin_missing[:, 0],
            "shot_follows_dribble":      X_bin[:, 1],
            "shot_follows_dribble_missing": X_bin_missing[:, 1],
            "under_pressure":            X_bin[:, 2],
            "under_pressure_missing":    X_bin_missing[:, 2],
            "is_open_goal":              X_bin[:, 3],
            "is_open_goal_missing":      X_bin_missing[:, 3],
            # pressure
            "pressure_idx":        pressure_idx,
            "pressure_missing":    pressure_missing,
            "pressure_levels":     levels,
            "n_pressure_levels":   n_pressure_levels,
            # categorical
            "cat_idx":        cat_idx,
            "cat_missing":    cat_missing,
            "cat_categories": cat_categories,
            "n_levels":       n_levels,
            # categorical missing arrays (used directly in model definition)
            "shot_body_part_missing": cat_missing["shot_body_part"],
            "shot_technique_missing": cat_missing["shot_technique"],
            "assist_type_missing":    cat_missing["assist_type"],
            "assist_height_missing":  cat_missing["assist_height"],
            "play_pattern_missing":   cat_missing["play_pattern"]
        }

        if target_col is not None:
            model_inputs["y"] = df[target_col].fillna(0).astype(int).to_numpy()

        return model_inputs

    def _transform_continuous(self, X: pd.DataFrame, fit_scaler: bool):
        cols = self.MODEL_VARIABLES["cont_cols"]
        missing = X[cols].isna().astype(float).to_numpy()

        if fit_scaler:
            self._scaler = StandardScaler()
            self._scaler.fit(X[cols])

        scaled = self._scaler.transform(X[cols])
        scaled = np.where(np.isnan(scaled), 0.0, scaled)
        return scaled, missing

    def _transform_binary(self, X: pd.DataFrame):
        cols = self.MODEL_VARIABLES["bin_cols"]
        missing = X[cols].isna().astype(float).to_numpy()

        Xb = (
            X[cols]
            .replace({"True": 1, "False": 0, "true": 1, "false": 0,
                      "TRUE": 1, "FALSE": 0, True: 1, False: 0})
            .apply(pd.to_numeric, errors="coerce")
            .fillna(-1.0)
            .to_numpy(dtype=float)
        )
        return Xb, missing

    def _transform_categorical(self, X: pd.DataFrame, fit_scaler: bool):
        cols = self.MODEL_VARIABLES["cat_cols"]
        cat_missing_df = X[cols].isna().astype(float)
        cat_idx = {}
        cat_missing = {}
        n_levels = {}
        cat_categories = {}

        for col in cols:
            if fit_scaler:
                known = [c for c in X[col].dropna().unique().tolist()]
                self._cat_categories = self._cat_categories or {}
                self._cat_categories[col] = known
            else:
                if not self._cat_categories or col not in self._cat_categories:
                    raise RuntimeError(
                        f"Model categories for '{col}' are not fitted. "
                        "Call fit() with training data first."
                    )
                known = self._cat_categories[col]

            cat = pd.Categorical(X[col], categories=known)
            codes = cat.codes.copy()
            unknown_idx = len(known)
            codes = np.where(codes == -1, unknown_idx, codes).astype(int)

            cat_idx[col]      = codes
            cat_missing[f"{col}"]  = cat_missing_df[col].to_numpy()
            n_levels[col]     = len(known) + 1          # +1 for unknown
            cat_categories[col] = known + ["Unknown"]
        return cat_idx, cat_missing, cat_categories, n_levels

    def _transform_pressure(self, X: pd.DataFrame, fit_scaler: bool):
        col = self.MODEL_VARIABLES["pressure_col"]
        series = X[col]
        missing = series.isna().astype(float).to_numpy()

        if fit_scaler:
            self._pressure_levels = sorted(
                [v for v in series.dropna().unique().tolist() if pd.notna(v)]
            )

        levels = self._pressure_levels
        if not levels:
            raise ValueError(f"'{col}' has no observed non-missing levels.")

        cat = pd.Categorical(series, categories=levels, ordered=True)
        codes = cat.codes.copy()
        unknown_idx = len(levels)
        codes = np.where(codes == -1, unknown_idx, codes).astype(int)

        return codes, missing, len(levels) + 1, levels